In [114]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import tensorflow as tf

from utils import column_encoder
from models import RecommenderModel

In [115]:
data = pd.read_csv('personalized_recommendation_dataset.csv')
data

,User_ID,Item_ID,Category,Rating,Timestamp,Price,Platform,Location
0,User_913,Item_52,Movies,2.0,2023-05-15,369.55,Web,Africa
1,User_3457,Item_66,Electronics,1.4,2023-08-19,255.15,Web,Africa
2,User_1629,Item_1467,Sports,2.7,2024-03-27,296.69,Web,Europe
3,User_3463,Item_697,Movies,1.6,2023-12-03,55.59,Tablet,North America
4,User_2941,Item_1736,Games,3.4,2023-02-06,366.22,Web,South America
...,...,...,...,...,...,...,...,...
149995,User_577,Item_1195,Sports,2.4,2023-10-24,182.04,Web,Europe
149996,User_4996,Item_1335,Books,3.4,2023-07-21,345.84,Mobile App,Europe
149997,User_2804,Item_1956,Games,1.6,2024-02-27,203.77,Mobile App,Europe
149998,User_1443,Item_397,Books,1.4,2024-01-23,475.06,Smart TV,Europe


In [116]:
print(data.info())
print(data.describe())
print(data.isnull().sum())
print(data.nunique())
print(f'Number Of Duplicates: {data.duplicated().sum()}')

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   User_ID    150000 non-null  str    
 1   Item_ID    150000 non-null  str    
 2   Category   150000 non-null  str    
 3   Rating     150000 non-null  float64
 4   Timestamp  150000 non-null  str    
 5   Price      150000 non-null  float64
 6   Platform   150000 non-null  str    
 7   Location   150000 non-null  str    
dtypes: float64(2), str(6)
memory usage: 9.2 MB
None
              Rating          Price
count  150000.000000  150000.000000
mean        2.994918     252.312944
std         1.153429     142.754120
min         1.000000       5.000000
25%         2.000000     128.817500
50%         3.000000     252.610000
75%         4.000000     375.630000
max         5.000000     500.000000
User_ID      0
Item_ID      0
Category     0
Rating       0
Timestamp    0
Price        0
Platform     0
Location 

# Preprocessing Data

In [117]:
columns = ['User_ID', 'Item_ID', 'Category', 'Platform', 'Location']

data, _ = column_encoder(data, *columns)
data

,User_ID,Item_ID,Category,Rating,Timestamp,Price,Platform,Location
0,4905,1468,6,2.0,2023-05-15,369.55,3,0
1,2731,1623,3,1.4,2023-08-19,255.15,3,0
2,700,520,8,2.7,2024-03-27,296.69,3,3
3,2738,1664,6,1.6,2023-12-03,55.59,2,4
4,2158,819,4,3.4,2023-02-06,366.22,3,5
...,...,...,...,...,...,...,...,...
149995,4531,218,8,2.4,2023-10-24,182.04,3,3
149996,4440,374,1,3.4,2023-07-21,345.84,0,3
149997,2006,1063,4,1.6,2024-02-27,203.77,0,3
149998,494,1331,1,1.4,2024-01-23,475.06,1,3



## Transforming Timestamp data
### to Days Passed since interaction

In [118]:
data['Timestamp'] = pd.to_datetime(data['Timestamp'])
data = data.sort_values('Timestamp')

In [119]:
time_delta = pd.Timestamp.now() - data['Timestamp']
time_delta = np.round(time_delta / pd.Timedelta(days=1), 0)

data['Days_since_interaction'] = time_delta

data

,User_ID,Item_ID,Category,Rating,Timestamp,Price,Platform,Location,Days_since_interaction
10773,3418,706,6,4.6,2022-12-24,175.34,3,0,1315.0
97910,2707,1051,8,4.8,2022-12-24,487.57,2,5,1315.0
13563,2962,634,3,1.6,2022-12-24,138.19,0,2,1315.0
34060,4674,363,1,4.1,2022-12-24,290.17,2,3,1315.0
34589,4647,886,5,3.3,2022-12-24,197.78,0,4,1315.0
...,...,...,...,...,...,...,...,...,...
64407,475,448,6,4.0,2024-12-23,155.66,3,3,585.0
68344,3243,1855,6,2.5,2024-12-23,141.37,2,3,585.0
144945,647,283,8,3.9,2024-12-23,435.80,3,2,585.0
29426,2974,30,7,1.8,2024-12-23,129.07,1,3,585.0


# Splitting Data

In [120]:
split_index = int(len(data) * 0.8)

train_data = data.iloc[:split_index].copy()
test_data = data.iloc[split_index:].copy()

In [121]:
price_scaler = StandardScaler()

train_data['Price'] = price_scaler.fit_transform(train_data['Price'].to_numpy().reshape(-1 ,1))
test_data['Price'] = price_scaler.transform(test_data['Price'].to_numpy().reshape(-1 ,1))

In [122]:
print(f'Train shape: {train_data.shape}')
print(f'Test shape: {test_data.shape}')

print(f'\nTrain Data Time Length: \n{train_data['Timestamp'].min()} --> {train_data['Timestamp'].max()}')
print(f'\nTest Data Time Length: \n{test_data['Timestamp'].min()} --> {test_data['Timestamp'].max()}')

Train shape: (120000, 9)
Test shape: (30000, 9)

Train Data Time Length: 
2022-12-24 00:00:00 --> 2024-07-30 00:00:00

Test Data Time Length: 
2024-07-30 00:00:00 --> 2024-12-23 00:00:00


In [123]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

from utils import column_encoder

In [124]:
data = pd.read_csv('personalized_recommendation_dataset.csv')
data

,User_ID,Item_ID,Category,Rating,Timestamp,Price,Platform,Location
0,User_913,Item_52,Movies,2.0,2023-05-15,369.55,Web,Africa
1,User_3457,Item_66,Electronics,1.4,2023-08-19,255.15,Web,Africa
2,User_1629,Item_1467,Sports,2.7,2024-03-27,296.69,Web,Europe
3,User_3463,Item_697,Movies,1.6,2023-12-03,55.59,Tablet,North America
4,User_2941,Item_1736,Games,3.4,2023-02-06,366.22,Web,South America
...,...,...,...,...,...,...,...,...
149995,User_577,Item_1195,Sports,2.4,2023-10-24,182.04,Web,Europe
149996,User_4996,Item_1335,Books,3.4,2023-07-21,345.84,Mobile App,Europe
149997,User_2804,Item_1956,Games,1.6,2024-02-27,203.77,Mobile App,Europe
149998,User_1443,Item_397,Books,1.4,2024-01-23,475.06,Smart TV,Europe


In [125]:
print(data.info())
print(data.describe())
print(data.isnull().sum())
print(data.nunique())
print(f'Number Of Duplicates: {data.duplicated().sum()}')

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   User_ID    150000 non-null  str    
 1   Item_ID    150000 non-null  str    
 2   Category   150000 non-null  str    
 3   Rating     150000 non-null  float64
 4   Timestamp  150000 non-null  str    
 5   Price      150000 non-null  float64
 6   Platform   150000 non-null  str    
 7   Location   150000 non-null  str    
dtypes: float64(2), str(6)
memory usage: 9.2 MB
None
              Rating          Price
count  150000.000000  150000.000000
mean        2.994918     252.312944
std         1.153429     142.754120
min         1.000000       5.000000
25%         2.000000     128.817500
50%         3.000000     252.610000
75%         4.000000     375.630000
max         5.000000     500.000000
User_ID      0
Item_ID      0
Category     0
Rating       0
Timestamp    0
Price        0
Platform     0
Location 

User_ID       5000
Item_ID       2000
Category         9
Rating          41
Timestamp      731
Price        47162
Platform         4
Location         6
dtype: int64
Number Of Duplicates: 0


# Preprocessing Data

In [126]:
columns = ['User_ID', 'Item_ID', 'Category', 'Platform', 'Location']

data, _ = column_encoder(data, *columns)
data


,User_ID,Item_ID,Category,Rating,Timestamp,Price,Platform,Location
0,4905,1468,6,2.0,2023-05-15,369.55,3,0
1,2731,1623,3,1.4,2023-08-19,255.15,3,0
2,700,520,8,2.7,2024-03-27,296.69,3,3
3,2738,1664,6,1.6,2023-12-03,55.59,2,4
4,2158,819,4,3.4,2023-02-06,366.22,3,5
...,...,...,...,...,...,...,...,...
149995,4531,218,8,2.4,2023-10-24,182.04,3,3
149996,4440,374,1,3.4,2023-07-21,345.84,0,3
149997,2006,1063,4,1.6,2024-02-27,203.77,0,3
149998,494,1331,1,1.4,2024-01-23,475.06,1,3



## Transforming Timestamp data
### to Days Passed since interaction

In [127]:
data['Timestamp'] = pd.to_datetime(data['Timestamp'])
data = data.sort_values('Timestamp')

In [128]:
price_scaler = StandardScaler()

train_data['Price'] = price_scaler.fit_transform(train_data['Price'].to_numpy().reshape(-1 ,1))
test_data['Price'] = price_scaler.transform(test_data['Price'].to_numpy().reshape(-1 ,1))

# Using Basic Collabrative Filtering 

In [129]:
columns = ['User_ID', 'Item_ID']
y = 'Rating'

train_ds = to_tensorflow_dataset(data, y, *columns)

# Train Dataset
train_ds_temp = tf.data.Dataset.from_tensor_slices(
    (
        {
        'User_ID' : train['User_ID'].values,
        'Item_ID' : train['Item_ID'].values
    },
    train['Rating'].values
    )
)

test_ds = tf.data.Dataset.from_tensor_slices(
    (
        {
        'User_ID' : test['User_ID'].values,
        'Item_ID' : test['Item_ID'].values
    },
    test['Rating'].values
    )
)

In [130]:
train_ds = train_ds.shuffle(10_000).batch(256)
test_ds = test_ds.batch(256)

In [131]:
model = RecommenderModel(
    num_users=num_users,
    num_items=num_items,
    feature_dim=10,
)

model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.001),
    loss = tf.keras.losses.MeanSquaredError()   
)

In [132]:
history = model.fit(train_ds, epochs=10, validation_data=test_ds)

Epoch 1/10


/home/kiamehr/.pyenv/versions/3.12.10/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


586/586 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - loss: 9.4152 - val_loss: 8.4787
Epoch 2/10
586/586 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - loss: 7.1213 - val_loss: 4.9918
Epoch 3/10
586/586 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 3.2225 - val_loss: 1.8327
Epoch 4/10
586/586 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 1.5455 - val_loss: 1.2886
Epoch 5/10
586/586 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: 1.3307 - val_loss: 1.2377
Epoch 6/10
586/586 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - loss: 1.3070 - val_loss: 1.2290
Epoch 7/10
586/586 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - loss: 1.3001 - val_loss: 1.2242
Epoch 8/10
586/586 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - loss: 1.2954 - val_loss: 1.2202
Epoch 9/10
586/586 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - loss: 1.2906 - val_loss: 1.2152
Epoch 10/10
586/586 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - loss: 1.2852 - val_loss: 1.2093


In [133]:
print(history.history)

{'loss': [9.415234565734863, 7.121335029602051, 3.222485065460205, 1.5454869270324707, 1.3307093381881714, 1.3070194721221924, 1.300115942955017, 1.2953729629516602, 1.290588617324829, 1.2852141857147217], 'val_loss': [8.478669166564941, 4.991820812225342, 1.8326916694641113, 1.2886430025100708, 1.2377128601074219, 1.2290009260177612, 1.2241984605789185, 1.2201541662216187, 1.2151883840560913, 1.2093276977539062]}


In [134]:
def y(*sex):
    print(sex)

x = ['kia', 'mmd']

y(*x)


('kia', 'mmd')
